# 07. Контролируемый опыт: full / spatial / temporal / combined

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Проверим гипотезу: tiling сохраняет формы на выбранной grid, но меняет результат нелокального encoder/decoder. Измерим разницу и CPU latency на трёх случайных tiny checkpoints.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT связь:** после VAE training (cells 70–74) зафиксируйте reconstruction baseline; после Q5.1 (76–77) отдельно оценивайте codec и DiT. Здесь benchmark codec, без generative ODE. [Критерии перехода](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='07_controlled_benchmark';metrics={}
print('Pinned source:',source_hash())


## 1. Гипотеза, контроль и метрики

**H1:** на coverage-safe размерах original tiling сохраняет BCTHW, но native tiny reconstruction отличается от full из-за context/padding/GroupNorm. **Контроль:** те же random weights, input и deterministic posterior mode. **Фактор:** только tile policy; три seeds показывают устойчивость свойства, а не оценивают pretrained distribution.

Все результаты сравниваются с full и отдельно с input. Для нормированных RGB x∈[−1,1] dynamic range R=2:

$$
\mathrm{MSE}=\frac{1}{N}\sum_i(x_i-y_i)^2,\qquad
\mathrm{PSNR}=10\log_{10}\frac{R^2}{\mathrm{MSE}}.
$$

PSNR к input — reconstruction quality **случайного tiny codec**; PSNR к full — deviation от baseline. Они отвечают на разные вопросы. На латенте R=2 произволен, поэтому в выводах используем latent MSE, а PSNR только для RGB signal convention.

Вместо одного среднего дополнительно измерим spatial seams vs interior и temporal profile. Peak GPU memory не измеряется в CPU опыте. Warmup исключён; median трёх timing repeats уменьшает шум, не даёт производственный benchmark.


In [ ]:
rows=[];visual={}
x=synthetic_video(33,48,48,seed=101)
policies={'full':(False,False),'spatial':(True,False),'temporal':(False,True),'combined':(True,True)}
for seed in [0,1,2]:
    codec=TinyCodec(seed)
    def reconstruct():
        moments=codec._encode(x)
        return codec._decode(DiagonalPosterior(moments).mode()).sample
    codec.set_tiles(spatial=False,temporal=False)
    baseline,_=measure(reconstruct,repeats=3)
    for name,(spatial,temporal) in policies.items():
        codec.set_tiles(spatial=spatial,temporal=temporal)
        out,timing=measure(reconstruct,repeats=3)
        assert out.shape==x.shape
        delta=(out-baseline).square().mean((0,1,2))
        seam=torch.zeros_like(delta,dtype=torch.bool)
        for boundary in [16,32]:
            seam[max(0,boundary-1):boundary+1,:]=True
            seam[:,max(0,boundary-1):boundary+1]=True
        row={'seed':seed,'policy':name,**timing,
             'difference_to_full':errors(baseline,out),'reconstruction_vs_input':errors(x,out),
             'seam_MSE':float(delta[seam].mean()),'interior_MSE':float(delta[~seam].mean())}
        rows.append(row)
        if seed==0:visual[name]=out.detach().clone()
metrics['runs']=rows
assert all(r['difference_to_full']['mse']==0 for r in rows if r['policy']=='full')
assert all(r['difference_to_full']['mse']>0 for r in rows if r['policy']!='full')
for r in rows:print(r['seed'],r['policy'],'MSE to full=',round(r['difference_to_full']['mse'],6),'CPU s=',round(r['seconds_median'],3))


## 2. Графики и диагностическая локализация

Первый график сопоставляет MSE и CPU time. Если tiny tiling медленнее full, это ожидаемо: повторные overlapping convolutions и Python loops. Это не доказывает замедление на real GPU или невозможность больших videos; цель tiling — вместимость, которую этот CPU опыт не измеряет.

Seam mask — двухpixel полоса вокруг nominal spatial boundaries, одинаковая для всех policies. Ненулевая interior error говорит, что расхождение не ограничено линией сшивки. Сравнение маленькой seam area и большой interior требует means по каждой области, не суммы.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,3.4))
for seed in [0,1,2]:
    rr=[r for r in rows if r['seed']==seed]
    axes[0].plot([r['policy'] for r in rr],[r['difference_to_full']['mse'] for r in rr],'o-',label=f'seed {seed}')
    axes[1].plot([r['policy'] for r in rr],[r['seconds_median'] for r in rr],'o-')
rr=[r for r in rows if r['seed']==0]
axes[2].plot([r['policy'] for r in rr],[r['seam_MSE'] for r in rr],'o-',label='Seam band')
axes[2].plot([r['policy'] for r in rr],[r['interior_MSE'] for r in rr],'x--',label='Interior')
axes[0].set(ylabel='RGB MSE to full',title='Identical input and weights');axes[0].legend()
axes[1].set(ylabel='CPU median seconds',title='Warmup + 3 timed runs')
axes[2].set(ylabel='RGB MSE to full',title='Seed 0 spatial localization');axes[2].legend()
for ax in axes:ax.tick_params(axis='x',rotation=25)
fig.tight_layout();fig.savefig(figure_path(LAB,'benchmark'));plt.show()
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,(name,out) in zip(axes,visual.items()):
    delta=(out-visual['full']).square().mean((0,1,2));im=ax.imshow(delta.numpy(),vmin=0,vmax=max(float((visual['combined']-visual['full']).square().mean((0,1,2)).max()),1e-9));ax.set(title=name,xlabel='W',ylabel='H')
fig.tight_layout();fig.savefig(figure_path(LAB,'benchmark_error_maps'));plt.show()


## 3. Overlap work и retained buffers

При большом regular 3D grid отношение обработанного объёма к полезному приблизительно:

$$
R_{work}\approx\frac{Q_t+1}{S_t}\cdot\frac{Q_h}{S_h}\cdot\frac{Q_w}{S_w}.
$$

Это оценка числа повторных input positions, не точная FLOPs/latency ratio: первый/последний tile, компрессия, attention, backend и launch overhead меняют стоимость. Здесь spatial Q=32,S=16 и temporal Q+1=17,S=8 дают большой overlap. Для реального codec размер tile и overlap выбирают по budget/качество/latency совместно.


In [ ]:
metrics['asymptotic_processed_volume_ratio']=(17/8)*(32/16)**2
metrics['conclusion']={
 'shape_invariance_on_test_grid':True,
 'tiling_is_full_equivalent':False,
 'native_mid_attention_tested':False,
 'pretrained_quality_tested':False,
 'peak_VRAM_tested':False,
 'CPU_timing_is_production_latency':False}
print(metrics['conclusion'])


## Упражнения и критерий перехода

1. Добавьте overlap=8/16 и сравните quality deviation/time при корректном coverage.
2. Выполните encoder-only и decoder-only ablations: какой из них вносит больше MSE?
3. Для pretrained phase выберите non-overlapping video split и бюджет повторов; не используйте seed как замену video diversity. Переход: сформулируйте вывод отдельно по гипотезе H1, latency и памяти.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
